In [0]:
#Importando as bibliotecas que serão utilizadas no notebook
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType
     

#Criando um sessão so spark para carregar os dados CSV
spark = SparkSession.builder.appName("olist-analysis").getOrCreate()
     

#Definindo o path das camadas do Lakehouse
pathRaw = "dbfs:/FileStore/project/olist/raw"
pathBronze = "dbfs:/FileStore/project/olist/bronze"
pathSilver = "dbfs:/FileStore/project/olist/silver"
pathGold = "dbfs:/FileStore/project/olist/gold"

In [0]:
# Carregando dados na Camada Silver

#Customers data
customersSilver = (
    spark.sql(f'''
       SELECT
            customerId,
            customerUniqueId,
            customerZipCodePrefix,
            customerCity,
            customerState,
            CAST(DataRawLoad AS TIMESTAMP) AS DataRawLoad
       FROM
          (
            SELECT 
                DENSE_RANK() OVER(ORDER BY DataRawLoad DESC) AS rank, * 
            FROM delta.`{pathBronze}/customers`
          ) AS C
       WHERE
            C.rank = 1
       ''')
)
(
    customersSilver
     .write
     .format("delta")
     .mode("overwrite")
     .save(f"{pathSilver}/customers")
)

# Geolocation
geolocationSilver = (
    spark.sql(f'''
       SELECT
            geolocationCodePrefix,
            geolocationLat,
            geolocationLng,
            geolocationCity,
            geolocationState,
            CAST(DataRawLoad AS TIMESTAMP) AS DataRawLoad
       FROM
          (
            SELECT 
                DENSE_RANK() OVER(ORDER BY DataRawLoad DESC) AS rank, * 
            FROM delta.`{pathBronze}/geolocation`
          ) AS G
       WHERE
            G.rank = 1
       ''')
)
(
geolocationSilver
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{pathSilver}/geolocation")
)

# Order_items
order_itemsSilver = (
    spark.sql(f'''
       SELECT
            orderId,
            orderItemId,
            productId,
            sellerId,
            shippingLimitDate,
            price,
            freightValue,
            CAST(DataRawLoad AS TIMESTAMP) AS DataRawLoad
       FROM
          (
            SELECT 
                DENSE_RANK() OVER(ORDER BY DataRawLoad DESC) AS rank, * 
            FROM delta.`{pathBronze}/order_items`
          ) AS O
       WHERE
            O.rank = 1
       ''')
)
(
order_itemsSilver
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{pathSilver}/order_items")
)

# Order Payments
order_paymentsSilver = (
    spark.sql(f'''
       SELECT
            orderId,
            paymentSequential,
            paymentType,
            paymentInstallments,
            paymentValue,
            CAST(DataRawLoad AS TIMESTAMP) AS DataRawLoad
       FROM
          (
            SELECT 
                DENSE_RANK() OVER(ORDER BY DataRawLoad DESC) AS rank, * 
            FROM delta.`{pathBronze}/order_payments`
          ) AS T
       WHERE
            T.rank = 1
       ''')
)
(
order_paymentsSilver
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{pathSilver}/order_payments")
)

order_reviewsSilver = (
    spark.sql(f'''
       SELECT
            reviewId,
            orderId,
            reviewScore,
            reviewCommentTitle,
            reviewCommentMessage,
            reviewCreationDate,
            reviewAnswerTimestamp,
            CAST(DataRawLoad AS TIMESTAMP) AS DataRawLoad
       FROM
          (
            SELECT 
                DENSE_RANK() OVER(ORDER BY DataRawLoad DESC) AS rank, * 
            FROM delta.`{pathBronze}/order_reviews`
          ) AS R
       WHERE
            R.rank = 1
       ''')
)
(
order_reviewsSilver
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{pathSilver}/order_reviews")
)

# Orders
ordersSilver = (
    spark.sql(f'''
       SELECT
            orderId,
            customerId,
            orderStatus,
            orderPurchaseTimestamp,
            orderApprovedAt,
            orderDeliveredCarrierDate,
            orderDeliveredCustomerDate,
            orderEstimatedDeliveryDate,
            CAST(DataRawLoad AS TIMESTAMP) AS DataRawLoad
       FROM
          (
            SELECT 
                DENSE_RANK() OVER(ORDER BY DataRawLoad DESC) AS rank, * 
            FROM delta.`{pathBronze}/orders`
          ) AS O
       WHERE
            O.rank = 1
       ''')
)
(
ordersSilver
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{pathSilver}/orders")
)

# Products
productsSilver = (
    spark.sql(f'''
       SELECT
            productId,
            productCategoryName,
            productNameLenght,
            productDescriptionLenght,
            productPhotosQty,
            productWeight_g,
            productLength_cm,
            productHeight_cm,
            productWidth_cm,
            CAST(DataRawLoad AS TIMESTAMP) AS DataRawLoad
       FROM
          (
            SELECT 
                DENSE_RANK() OVER(ORDER BY DataRawLoad DESC) AS rank, * 
            FROM delta.`{pathBronze}/products`
           ) AS P
       WHERE
            P.rank = 1
       ''')
)
(
productsSilver
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{pathSilver}/products")
)

# Sellers
sellersSilver = (
    spark.sql(f'''
       SELECT
            sellerId,
            sellerCodePrefix,
            sellerCity,
            sellerState,
            CAST(DataRawLoad AS TIMESTAMP) AS DataRawLoad
       FROM
          (
            SELECT 
                DENSE_RANK() OVER(ORDER BY DataRawLoad DESC) AS rank, * 
            FROM delta.`{pathBronze}/sellers`
          ) AS T
       WHERE
            T.rank = 1
       ''')
)
(
sellersSilver
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{pathSilver}/sellers")
)

# Category
categorySilver = (
    spark.sql(f'''
       SELECT
            productCategory,
            productCategoryNameEnglish,
            CAST(DataRawLoad AS TIMESTAMP) AS DataRawLoad
       FROM
          (
            SELECT 
                dense_rank() over(order by DataRawLoad desc) as rank, * 
            FROM delta.`{pathBronze}/category`
          ) AS C
       WHERE
            C.rank = 1
        ''')
)
(
categorySilver
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{pathSilver}/category")
)